# PROTOCOL: 4-Channel + Attention MIL
High-Resolution Feature Extraction with Attention-Based Multiple Instance Learning (AB-MIL).
### Key Architectural Features
* **4-Channel Input:** Standard RGB + **Stain-Normalized Hematoxylin Channel** (via Macenko deconvolution) to explicitly provide nuclear structure.
* **High-Resolution Patching:** Training performed on random **$512 \times 512$ patches** from full-resolution slides to preserve chromatin texture details, guided by **Otsu Thresholding** for tissue relevance.
* **Backbone:** **ConvNeXt-Tiny** modified for 4-channel input, utilized as a feature extractor for high-fidelity tissue patches.
* **Aggregation:** **Attention-Based MIL** replaces simple averaging and LightGBM. A **Gated Attention Network** learns to assign importance weights to each tile, dynamically filtering out healthy tissue noise and focusing on cancerous regions.
* **Optimization:** Employing the **Lion Optimizer** ($LR=5e^{-5}$) with Cosine Annealing and **MixUp Augmentation** to improve generalization on limited data.
* **Validation:** 5-Fold Stratified Cross-Validation ensuring robust performance assessment.

### Inference Pipeline
* **Phase 1 (Vision):** ConvNeXt-Tiny backbone extracts 768-dimensional feature vectors for every tile in the slide.
* **Phase 2 (Logic):** The MIL head aggregates these feature bags into a slide-level prediction, capable of identifying sparse signals (e.g., small focal tumors).
* **Oracle ROI Extraction:** Uses ground-truth masks for deterministic cropping, removing the need for a regressor.

In [ ]:
# ============================================================
# 1. Imports and Configuration
# ============================================================
import os
import random
import warnings
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
from PIL import Image
import tifffile as tiff

import torch
from torch import nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import torchvision.transforms.functional as TF
from torch.cuda.amp import autocast, GradScaler

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, classification_report
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.manifold import TSNE

warnings.filterwarnings("ignore")
sns.set(font_scale=1.2)
sns.set_style("whitegrid")

class Config:
    SEED = 42
    MODEL_NAME = "convnext_tiny"
    
    # --- PATHS ---
    # CLEAN_IMG_DIR: Processed training images (Goo removed)
    CLEAN_IMG_DIR = '/kaggle/input/dataset/train_data_cleaned/images'
    CLEAN_MASK_DIR = '/kaggle/input/dataset/train_data_cleaned/masks'
    CLEAN_LABELS  = '/kaggle/input/dataset/train_data_cleaned/train_labels.csv'
    
    # TIFF_DIR: Macenko-Normalized Hematoxylin Channels (Masked)
    TRAIN_TIFF_DIR = '/kaggle/input/tiff-images/train' 
    TEST_TIFF_DIR = '/kaggle/input/tiff-images/test' 
    
    # TEST PATHS
    TEST_IMG_DIR = '/kaggle/input/dataset/test_data/images'
    TEST_MASK_DIR = '/kaggle/input/dataset/test_data/masks'
    

    # --- HYPERPARAMS ---
    IMG_SIZE_CLS = 512   # High-Res Patch Size
    STRIDE = 256         # Overlap for Inference Tiling
    BATCH_SIZE = 16      
    NUM_CLASSES = 4
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # OPTIMIZATION
    EPOCHS_CLS = 50      
    ACCUM_STEPS = 2      
    LR_CLS = 5e-5        # Low LR for Lion
    WD_CLS = 1e-2        # High Decay for Lion
    K_FOLDS = 5

def set_seed(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    
set_seed(Config.SEED)
print(f"Operational Device: {Config.DEVICE}")


# ============================================================
# 2. Utilities & Optimizers
# ============================================================

class Lion(torch.optim.Optimizer):
    def __init__(self, params, lr=1e-4, betas=(0.9, 0.99), weight_decay=0.0):
        if not 0.0 <= lr: raise ValueError(f"Invalid learning rate: {lr}")
        if not 0.0 <= betas[0] < 1.0: raise ValueError(f"Invalid beta parameter at index 0: {betas[0]}")
        if not 0.0 <= betas[1] < 1.0: raise ValueError(f"Invalid beta parameter at index 1: {betas[1]}")
        defaults = dict(lr=lr, betas=betas, weight_decay=weight_decay)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad(): loss = closure()
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None: continue
                p.data.mul_(1 - group['lr'] * group['weight_decay'])
                grad = p.grad
                state = self.state[p]
                if len(state) == 0: state['exp_avg'] = torch.zeros_like(p)
                exp_avg = state['exp_avg']
                beta1, beta2 = group['betas']
                update = exp_avg * beta1 + grad * (1 - beta1)
                p.add_(torch.sign(update), alpha=-group['lr'])
                exp_avg.mul_(beta2).add_(grad, alpha=1 - beta2)
        return loss

class EarlyStopping:
    def __init__(self, patience=7, mode="max", verbose=False, delta=0, path='checkpoint.pth', trace_func=print):
        self.patience = patience
        self.mode = mode
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.delta = delta
        self.path = path
        self.trace_func = trace_func

    def __call__(self, val_metric, model):
        score = val_metric
        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_metric, model)
            return

        improved = False
        if self.mode == "min":
            if score < (self.best_score - self.delta): improved = True
        elif self.mode == "max":
            if score > (self.best_score + self.delta): improved = True
        
        if improved:
            self.best_score = score
            self.save_checkpoint(val_metric, model)
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience: self.early_stop = True

    def save_checkpoint(self, val_metric, model):
        if self.verbose: self.trace_func(f'   ✅ Improved ({self.best_score:.4f}). Saving...')
        torch.save(model.state_dict(), self.path)

# Helper: Mixup
def mixup_data(x, y, alpha=0.4):
    if alpha > 0: lam = np.random.beta(alpha, alpha)
    else: lam = 1
    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(Config.DEVICE)
    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

def get_bbox_from_mask(mask_np):
    rows = np.any(mask_np, axis=1)
    cols = np.any(mask_np, axis=0)
    if not np.any(rows) or not np.any(cols): return None
    ymin, ymax = np.where(rows)[0][[0, -1]]
    xmin, xmax = np.where(cols)[0][[0, -1]]
    return xmin, ymin, xmax, ymax

def extract_embeddings(model, x):
    """Generic feature extraction for different architectures."""
    name = Config.MODEL_NAME.lower()
    
    if name == "convnext_tiny":
        x = model.features(x)
        x = model.avgpool(x)
        x = model.classifier[0](x) # LayerNorm
        x = model.classifier[1](x) # Flatten
        return x
    # ... (Add other architectures if needed) ...
    else: return model.features(x) # Generic fallback


In [ ]:
# ============================================================
# 3. Dataset (Mosaic & Smart Patching)
# ============================================================
class MosaicDataset(Dataset):
    def __init__(self, df, img_dir, mask_dir, tiff_dir, transform=None, mode='train', 
                 patches_per_image=1, cache_data=True):
        self.df = df
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.tiff_dir = tiff_dir
        self.mode = mode
        self.patches_per_image = patches_per_image 
        self.le = LabelEncoder()
        
        self.cache = []
        if 'label' in self.df.columns:
            self.df['encoded_label'] = self.le.fit_transform(self.df['label'])
            
        if cache_data:
            print(f"   Pre-loading {len(df)} images...")
            for idx in tqdm(range(len(df)), desc="Caching"):
                self.cache.append(self.load_item(idx))

    def __len__(self):
        return len(self.df) * self.patches_per_image

    def load_item(self, idx):
        row = self.df.iloc[idx]
        fname = row['sample_index']
        
        img_path = os.path.join(self.img_dir, fname)
        rgb = cv2.imread(img_path)
        rgb = cv2.cvtColor(rgb, cv2.COLOR_BGR2RGB)
        
        tiff_name = fname.replace(".png", ".tif")
        tiff_path = os.path.join(self.tiff_dir, tiff_name)
        if os.path.exists(tiff_path):
            tif = tiff.imread(tiff_path)
        else:
            tif = np.zeros(rgb.shape[:2], dtype=np.uint8)
            
        if tif.ndim == 2: tif = np.expand_dims(tif, axis=-1)
        
        # Tissue Guidance Mask
        mask_name = fname.replace("img_", "mask_")
        mask_path = None
        if os.path.exists(os.path.join(self.mask_dir, mask_name)): 
            mask_path = os.path.join(self.mask_dir, mask_name)
            
        if mask_path: mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        else:
            hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
            _, mask = cv2.threshold(hsv[:,:,1], 0, 255, cv2.THRESH_BINARY+cv2.THRESH_OTSU)

        label = row['encoded_label'] if 'encoded_label' in row else -1
        return rgb, tif, mask, label

    def get_tissue_guided_crop(self, rgb, tif, mask, crop_size=512):
        h, w = rgb.shape[:2]
        if h < crop_size or w < crop_size:
            pad_h = (crop_size - h % crop_size) % crop_size
            pad_w = (crop_size - w % crop_size) % crop_size
            rgb = np.pad(rgb, ((0, pad_h), (0, pad_w), (0, 0)))
            tif = np.pad(tif, ((0, pad_h), (0, pad_w), (0, 0)))
            mask = np.pad(mask, ((0, pad_h), (0, pad_w)))
            h, w = rgb.shape[:2]

        for _ in range(10):
            y = random.randint(0, h - crop_size)
            x = random.randint(0, w - crop_size)
            mask_crop = mask[y:y+crop_size, x:x+crop_size]
            if (np.count_nonzero(mask_crop)/(crop_size**2)) > 0.30: 
                return rgb[y:y+crop_size, x:x+crop_size], tif[y:y+crop_size, x:x+crop_size]
        
        cy, cx = h//2, w//2
        y, x = max(0, cy-crop_size//2), max(0, cx-crop_size//2)
        return rgb[y:y+crop_size, x:x+crop_size], tif[y:y+crop_size, x:x+crop_size]

    def __getitem__(self, idx):
        real_idx = idx % len(self.df)
        rgb, tif, mask, label = self.cache[real_idx] if self.cache else self.load_item(real_idx)
        
        rgb_patch, tif_patch = self.get_tissue_guided_crop(rgb, tif, mask, Config.IMG_SIZE_CLS)
        
        rgb_t = TF.to_tensor(rgb_patch)
        tif_t = TF.to_tensor(tif_patch)
        full_t = torch.cat([rgb_t, tif_t], dim=0)

        if self.mode == 'train':
            if random.random() > 0.5: full_t = TF.hflip(full_t)
            if random.random() > 0.5: full_t = TF.vflip(full_t)
            if random.random() > 0.5: full_t = TF.rotate(full_t, random.choice([90, 180, 270]))
            
            if random.random() > 0.2:
                full_t[:3] = TF.adjust_brightness(full_t[:3], 1 + (random.random()*0.4 - 0.2))
                full_t[:3] = TF.adjust_contrast(full_t[:3], 1 + (random.random()*0.4 - 0.2))
                full_t[3] = full_t[3] * (1.0 + (random.random() * 0.4 - 0.2))
                full_t = torch.clamp(full_t, 0, 1)
            
            if random.random() > 0.3: transforms.RandomErasing(p=1.0, scale=(0.02, 0.15))(full_t)

        mean, std = [0.485, 0.456, 0.406, 0.5], [0.229, 0.224, 0.225, 0.5]
        full_t = TF.normalize(full_t, mean=mean, std=std)
        return full_t, label


In [ ]:
# ============================================================
# 4. Phase 1 Model (ConvNeXt)
# ============================================================
def get_classifier_model(num_classes):
    print("   Instantiating ConvNeXt-Tiny (4-Channel)...")
    model = models.convnext_tiny(weights='IMAGENET1K_V1')
    
    # 4-Channel Surgery
    old_conv = model.features[0][0]
    new_conv = nn.Conv2d(4, old_conv.out_channels, kernel_size=old_conv.kernel_size, 
                         stride=old_conv.stride, padding=old_conv.padding, bias=old_conv.bias is not None)
    with torch.no_grad():
        new_conv.weight[:, :3] = old_conv.weight
        new_conv.weight[:, 3:4] = torch.mean(old_conv.weight, dim=1, keepdim=True)
        if old_conv.bias is not None: new_conv.bias = nn.Parameter(old_conv.bias.clone())

    model.features[0][0] = new_conv
    model.classifier[2] = nn.Linear(model.classifier[2].in_features, num_classes)
    return model.to(Config.DEVICE)


In [ ]:
# ============================================================
# 5. Phase 2 Model (Attention MIL)
# ============================================================
class AttentionMIL(nn.Module):
    def __init__(self, in_features=768, num_classes=4, hidden_dim=256):
        super().__init__()
        self.L = in_features
        self.D = hidden_dim
        self.K = 1
        
        self.attention_V = nn.Sequential(nn.Linear(self.L, self.D), nn.Tanh())
        self.attention_U = nn.Sequential(nn.Linear(self.L, self.D), nn.Sigmoid())
        self.attention_weights = nn.Linear(self.D, self.K)
        
        self.classifier = nn.Sequential(nn.Linear(self.L * self.K, num_classes))

    def forward(self, x):
        # x: (1, N_Tiles, 768)
        x = x.squeeze(0) # (N_Tiles, 768)
        
        A_V = self.attention_V(x)
        A_U = self.attention_U(x)
        A = self.attention_weights(A_V * A_U) # N x 1
        A = torch.transpose(A, 1, 0)  # 1 x N
        A = F.softmax(A, dim=1) 
        
        M = torch.mm(A, x) # (1, 768)
        logits = self.classifier(M)
        return logits, A


In [ ]:
# ============================================================
# 6. Training Loop (Phase 1: Deep Learning)
# ============================================================
def train_classifier_kfold(df):
    print(f"\nPHASE 1: Vision Training ({Config.K_FOLDS}-Fold CV)...")
    skf = StratifiedKFold(n_splits=Config.K_FOLDS, shuffle=True, random_state=Config.SEED)
    le = LabelEncoder()
    df['encoded_label'] = le.fit_transform(df['label'])
    
    fold_histories = []
    oof_preds, oof_targets = [], []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(df, df['encoded_label'])):
        print(f"\nSTARTING FOLD {fold+1}/{Config.K_FOLDS}")
        train_df, val_df = df.iloc[train_idx].copy(), df.iloc[val_idx].copy()
        
        train_ds = MosaicDataset(train_df, Config.CLEAN_IMG_DIR, Config.CLEAN_MASK_DIR, Config.TRAIN_TIFF_DIR, mode='train', patches_per_image=4, cache_data=False)
        val_ds   = MosaicDataset(val_df,   Config.CLEAN_IMG_DIR, Config.CLEAN_MASK_DIR, Config.TRAIN_TIFF_DIR, mode='val', patches_per_image=1, cache_data=False)
        
        train_loader = DataLoader(train_ds, batch_size=Config.BATCH_SIZE, shuffle=True, num_workers=4, persistent_workers=True, pin_memory=True)
        val_loader   = DataLoader(val_ds,   batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=4, persistent_workers=True,  pin_memory=True)
        
        model = get_classifier_model(Config.NUM_CLASSES)
        
        all_labels = train_df['encoded_label'].values
        class_weights = torch.tensor(compute_class_weight('balanced', classes=np.unique(all_labels), y=all_labels), dtype=torch.float).to(Config.DEVICE)
        criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
        scaler = torch.cuda.amp.GradScaler()

        # Warmup
        print(f"   Fold {fold+1}: Warming up...")
        for param in model.parameters(): param.requires_grad = False
        for param in model.features[0][0].parameters(): param.requires_grad = True
        for param in model.classifier[2].parameters(): param.requires_grad = True
        
        optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)
        model.train()
        for epoch in range(5):
            for i, (imgs, lbls) in enumerate(train_loader):
                imgs, lbls = imgs.to(Config.DEVICE), lbls.to(Config.DEVICE)
                optimizer.zero_grad()
                with torch.cuda.amp.autocast():
                    loss = criterion(model(imgs), lbls) / Config.ACCUM_STEPS
                scaler.scale(loss).backward()
                if (i+1)%Config.ACCUM_STEPS==0: scaler.step(optimizer); scaler.update(); optimizer.zero_grad()
        
        # Full Training
        print(f"   Fold {fold+1}: Full Training...")
        for param in model.parameters(): param.requires_grad = True
        optimizer = Lion(model.parameters(), lr=Config.LR_CLS, weight_decay=Config.WD_CLS)
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=Config.EPOCHS_CLS, eta_min=1e-6)
        
        model_path = f"classifier_fold{fold}.pth"
        early_stopping = EarlyStopping(patience=15, mode="max", path=model_path)
        history = {
            'train_loss':[], 
            'train_acc':[], 
            'train_f1':[], 
            'val_loss':[], 
            'val_acc':[], 
            'val_f1':[]
        }
        
        for epoch in range(Config.EPOCHS_CLS):
            model.train()
            t_loss_sum, t_total, t_corr = 0, 0, 0
            optimizer.zero_grad()
            
            for i, (imgs, lbls) in enumerate(tqdm(train_loader, desc=f"Ep{epoch+1}", leave=False)):
                imgs, lbls = imgs.to(Config.DEVICE), lbls.to(Config.DEVICE)
                if random.random() < 0.5:
                    imgs, lbls_a, lbls_b, lam = mixup_data(imgs, lbls)
                    with torch.cuda.amp.autocast():
                        out = model(imgs)
                        loss = mixup_criterion(criterion, out, lbls_a, lbls_b, lam) / Config.ACCUM_STEPS
                else:
                    with torch.cuda.amp.autocast():
                        out = model(imgs)
                        loss = criterion(out, lbls) / Config.ACCUM_STEPS
                
                scaler.scale(loss).backward()
                if (i+1)%Config.ACCUM_STEPS==0: scaler.step(optimizer); scaler.update(); optimizer.zero_grad()
                
                t_loss_sum += loss.item() * Config.ACCUM_STEPS * imgs.size(0)
                _, preds = torch.max(out, 1)
                t_corr += (preds == lbls).sum().item(); t_total += lbls.size(0)
            
            scheduler.step()
            
            # Train F1
            t_preds, t_lbls = [], []
            model.eval()
            with torch.no_grad():
                for imgs, lbls in train_loader:
                    t_preds.extend(torch.argmax(model(imgs.to(Config.DEVICE)),1).cpu().numpy())
                    t_lbls.extend(lbls.numpy())
            t_f1 = f1_score(t_lbls, t_preds, average='macro')
            
            # Validation
            v_loss_sum, v_total, v_corr = 0, 0, 0
            v_preds, v_lbls = [], []
            with torch.no_grad():
                for imgs, lbls in val_loader:
                    imgs, lbls = imgs.to(Config.DEVICE), lbls.to(Config.DEVICE)
                    out = model(imgs)
                    loss = criterion(out, lbls)
                    v_loss_sum += loss.item() * imgs.size(0)
                    _, preds = torch.max(out, 1)
                    v_corr += (preds == lbls).sum().item(); v_total += lbls.size(0)
                    v_preds.extend(preds.cpu().numpy()); v_lbls.extend(lbls.cpu().numpy())
            
            t_loss, t_acc = t_loss_sum/t_total, t_corr/t_total
            v_loss, v_acc = v_loss_sum/v_total, v_corr/v_total
            v_f1 = f1_score(v_lbls, v_preds, average='macro')
            
            for k, v in zip(['train_loss','train_acc','train_f1','val_loss','val_acc','val_f1'], [t_loss, t_acc, t_f1, v_loss, v_acc, v_f1]): history[k].append(v)
            print(f"   Ep {epoch+1:3d}: T_Loss {t_loss:.4f} | T_F1 {t_f1:.4f} | V_Loss {v_loss:.4f} | V_F1 {v_f1:.4f}")
            
            early_stopping(v_f1, model)
            if early_stopping.early_stop: break
        
        fold_histories.append(history)
        
        # OOF
        print(f"   📊 Generating OOF Data...")
        model.load_state_dict(torch.load(model_path))
        model.eval()
        with torch.no_grad():
            for imgs, lbls in val_loader:
                imgs, lbls = imgs.to(Config.DEVICE), lbls.to(Config.DEVICE)
                oof_preds.extend(torch.argmax(model(imgs), 1).cpu().numpy())
                oof_targets.extend(lbls.cpu().numpy())
                
        del model, optimizer, scaler, train_loader, val_loader; torch.cuda.empty_cache()

    return le, fold_histories, f1_score(oof_targets, oof_preds, average='macro'), oof_targets, oof_preds


In [ ]:
# ============================================================
# 7. Phase 2: Attention MIL (Aggregation)
# ============================================================
def extract_raw_features(model, img_paths, tiff_dir, crop_size=512, stride=256):
    print(f"   Extracting RAW Deep Features...")
    model.eval()
    slide_features, labels, filenames = [], [], []
    mean_4ch, std_4ch = [0.485, 0.456, 0.406, 0.5], [0.229, 0.224, 0.225, 0.5]

    for fname in tqdm(img_paths, desc="Extracting"):
        rgb = cv2.imread(fname)
        if rgb is None: continue
        rgb = cv2.cvtColor(rgb, cv2.COLOR_BGR2RGB)
        
        tiff_path = os.path.join(tiff_dir, os.path.basename(fname).replace(".png", ".tif"))
        if os.path.exists(tiff_path):
            try:
                t = tiff.imread(tiff_path)
                if t.dtype!=np.uint8: t=(t/t.max()*255).astype(np.uint8)
            except: t=np.zeros(rgb.shape[:2], dtype=np.uint8)
        else: t=np.zeros(rgb.shape[:2], dtype=np.uint8)
        if t.ndim==2: t=np.expand_dims(t, axis=-1)
        
        full_img = np.concatenate([rgb, t], axis=-1)
        h, w = full_img.shape[:2]
        pad_h, pad_w = (crop_size - h%crop_size)%crop_size, (crop_size - w%crop_size)%crop_size
        if pad_h>0 or pad_w>0: full_img = np.pad(full_img, ((0, pad_h), (0, pad_w), (0, 0)))
        
        patches = []
        for y in range(0, h-crop_size+1, stride):
            for x in range(0, w-crop_size+1, stride):
                patch = full_img[y:y+crop_size, x:x+crop_size, :]
                patches.append(TF.normalize(TF.to_tensor(patch), mean=mean_4ch, std=std_4ch))
        
        if not patches: continue
        
        tile_feats = []
        with torch.no_grad():
            for i in range(0, len(patches), 32):
                batch = torch.stack(patches[i:i+32]).to(Config.DEVICE)
                x = extract_embeddings(model, batch)
                tile_feats.append(x.cpu().numpy())
                
        slide_features.append(np.concatenate(tile_feats, axis=0))
        filenames.append(os.path.basename(fname))
            
    return slide_features, filenames

def train_attention_mil(train_bags, train_labels, test_bags, n_folds=5):
    print(f"\n🧠 Training Attention MIL ({n_folds}-Fold)...")
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
    oof_preds = np.zeros((len(train_bags), 4))
    test_preds = np.zeros((len(test_bags), 4))
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(train_bags, train_labels)):
        mil_model = AttentionMIL(num_classes=4).to(Config.DEVICE)
        optimizer = torch.optim.Adam(mil_model.parameters(), lr=1e-4, weight_decay=1e-4)
        criterion = nn.CrossEntropyLoss()
        
        for epoch in range(20):
            mil_model.train()
            for idx in train_idx:
                bag = torch.from_numpy(train_bags[idx]).float().unsqueeze(0).to(Config.DEVICE)
                label = torch.tensor([train_labels[idx]]).to(Config.DEVICE)
                optimizer.zero_grad()
                logits, _ = mil_model(bag)
                loss = criterion(logits, label)
                loss.backward(); optimizer.step()
                
        mil_model.eval()
        with torch.no_grad():
            for idx in val_idx:
                bag = torch.from_numpy(train_bags[idx]).float().unsqueeze(0).to(Config.DEVICE)
                oof_preds[idx] = F.softmax(mil_model(bag)[0], dim=1).cpu().numpy()
            for i, bag in enumerate(test_bags):
                b = torch.from_numpy(bag).float().unsqueeze(0).to(Config.DEVICE)
                test_preds[i] += F.softmax(mil_model(b)[0], dim=1).cpu().numpy()/n_folds
                
    avg_f1 = f1_score(train_labels, np.argmax(oof_preds, axis=1), average='macro')
    print(f"✅ MIL CV Complete. Avg F1: {avg_f1:.4f}")
    return oof_preds, test_preds, avg_f1


In [ ]:
# ============================================================
# 8. Visualization
# ============================================================
def plot_history_cls(fold_histories):
    print(f"\n📊 Plotting History for {len(fold_histories)} Folds...")
    fig, axes = plt.subplots(1, 4, figsize=(26, 5))
    metrics = ['train_loss', 'train_acc', 'train_f1', 'val_f1']
    titles = ['Loss', 'Accuracy', 'Train F1', 'Val F1']
    
    for i, m in enumerate(metrics):
        ax = axes[i]
        max_len = max(len(h[m]) for h in fold_histories)
        for h in fold_histories:
            ax.plot(h[m], alpha=0.15)
        
        data = np.array([h[m] + [np.nan]*(max_len-len(h[m])) for h in fold_histories])
        ax.plot(np.nanmean(data, axis=0), linewidth=2, label='Mean')
        ax.set_title(titles[i]); ax.legend(); ax.grid(alpha=0.3)
    plt.show()

def plot_oof_confusion_matrix(y_true, y_pred, class_names, title_prefix="DL Baseline"):
    print(f"\n📊 {title_prefix} Confusion Matrix")
    if y_pred.ndim > 1: y_pred = np.argmax(y_pred, axis=1)
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
    plt.title(f"{title_prefix}"); plt.show()
    print(classification_report(y_true, y_pred, target_names=class_names))

def visualize_feature_space(model, loader, le, device):
    print("\n🎨 t-SNE Feature Space...")
    model.eval()
    feats, lbls = [], []
    with torch.no_grad():
        for imgs, l in tqdm(loader, desc="TSNE"):
            feats.append(extract_embeddings(model, imgs.to(device)).cpu().numpy())
            lbls.append(l.numpy())
    feats = np.concatenate(feats); lbls = np.concatenate(lbls)
    
    tsne = TSNE(n_components=2, perplexity=30, init='pca', learning_rate='auto', random_state=42)
    embedding = tsne.fit_transform(feats)
    
    plt.figure(figsize=(10, 8))
    df = pd.DataFrame(embedding, columns=['D1', 'D2'])
    df['L'] = le.inverse_transform(lbls)
    sns.scatterplot(data=df, x='D1', y='D2', hue='L', palette='tab10', s=20, alpha=0.7)
    plt.title("ConvNeXt Feature Space"); plt.show()

def run_visual_demo(model, le, test_dir, mask_dir, tiff_dir):
    print("\n📊 Visual Demo (Oracle Mode)...")
    model.eval()
    files = sorted([f for f in os.listdir(test_dir) if f.startswith('img_')])
    demo_files = random.sample(files, 10)
    
    imgs, boxes, probs = [], [], []
    for f in demo_files:
        img_p = os.path.join(test_dir, f)
        rgb = Image.open(img_p).convert("RGB")
        imgs.append(np.array(rgb))
        
        # Mask Logic
        m_name = f.replace("img_", "mask_") if os.path.exists(os.path.join(mask_dir, f.replace("img_", "mask_"))) else f.replace(".png", "_mask.png")
        m_path = os.path.join(mask_dir, m_name)
        
        if os.path.exists(m_path):
            m = np.array(Image.open(m_path).convert("L"))
            bbox = get_bbox_from_mask(m)
            if bbox: x1, y1, x2, y2 = bbox
            else: x1, y1, x2, y2 = 0, 0, rgb.width, rgb.height
        else: x1, y1, x2, y2 = 0, 0, rgb.width, rgb.height
        
        boxes.append([x1/rgb.width, y1/rgb.height, x2/rgb.width, y2/rgb.height])
        
        # Predict
        crop = rgb.crop((x1, y1, x2, y2)).resize((Config.IMG_SIZE_CLS, Config.IMG_SIZE_CLS))
        full = np.concatenate([np.array(crop), np.zeros((Config.IMG_SIZE_CLS, Config.IMG_SIZE_CLS, 1))], axis=-1)
        t = TF.normalize(TF.to_tensor(full), mean=[0.485, 0.456, 0.406, 0.5], std=[0.229, 0.224, 0.225, 0.5]).unsqueeze(0).to(Config.DEVICE)
        
        with torch.no_grad(): probs.append(F.softmax(model(t), dim=1).cpu().numpy()[0])
            
    # Visualization Grid
    fig, axes = plt.subplots(2, 5, figsize=(20, 8))
    axes = axes.flatten()
    for i in range(10):
        ax = axes[i]; img = imgs[i].copy(); h, w = img.shape[:2]
        x1, y1, x2, y2 = int(boxes[i][0]*w), int(boxes[i][1]*h), int(boxes[i][2]*w), int(boxes[i][3]*h)
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 3)
        idx = np.argmax(probs[i])
        ax.imshow(img); ax.set_title(f"{le.classes_[idx]} {probs[i][idx]:.2f}"); ax.axis('off')
    plt.tight_layout(); plt.show()


In [ ]:
# ============================================================
# 9. Main Execution
# ============================================================
if __name__ == "__main__":
    if os.path.exists(Config.CLEAN_LABELS):
        df = pd.read_csv(Config.CLEAN_LABELS)
        
        # 1. PHASE 1: Deep Learning Baseline
        le, histories, dl_f1, oof_targets, dl_preds = train_classifier_kfold(df)
        
        print("\n--- PHASE 1 DIAGNOSTICS ---")
        plot_history_cls(histories)
        plot_oof_confusion_matrix(oof_targets, dl_preds, le.classes_, "Phase 1 (DL)")
        
        # Viz
        model = get_classifier_model(Config.NUM_CLASSES)
        model.load_state_dict(torch.load("classifier_fold0.pth"))
        model = model.to(Config.DEVICE)
        viz_loader = DataLoader(MosaicDataset(df.iloc[:200], Config.CLEAN_IMG_DIR, Config.CLEAN_MASK_DIR, Config.TRAIN_TIFF_DIR, mode='val', patches_per_image=1, cache_data=False), batch_size=Config.BATCH_SIZE, shuffle=False)
        visualize_feature_space(model, viz_loader, le, Config.DEVICE)
        
        # 2. PHASE 2: AB-MIL
        print("\n--- PHASE 2: ATTENTION MIL ---")
        train_paths = [os.path.join(Config.CLEAN_IMG_DIR, f) for f in df['sample_index']]
        train_bags, train_names = extract_raw_features(model, train_paths, Config.TRAIN_TIFF_DIR)
        
        test_files = sorted([f for f in os.listdir(Config.TEST_IMG_DIR) if f.startswith("img_")])
        test_paths = [os.path.join(Config.TEST_IMG_DIR, f) for f in test_files]
        test_bags, test_names = extract_raw_features(model, test_paths, Config.TEST_TIFF_DIR)
        
        # Align labels
        y_train = le.transform(df.set_index('sample_index').loc[train_names]['label'].values)
        
        mil_oof, mil_test_preds, mil_f1 = train_attention_mil(train_bags, y_train, test_bags)
        
        plot_oof_confusion_matrix(y_train, mil_oof, le.classes_, "Phase 2 (MIL)")
        
        # 3. SUBMISSION
        final_preds = np.argmax(mil_test_preds, axis=1)
        final_labels = le.inverse_transform(final_preds)
        sub = pd.DataFrame({
            'sample_index': test_names, 
            'label': final_labels})
        f1_str = f"{mil_f1:.4f}".replace(".", "")
        sub.to_csv(f"submission_mil_f1_{f1_str}.csv", index=False)
        print(f"✅ Submission Saved!")
        
        # Demo
        run_visual_demo(model, le, Config.TEST_IMG_DIR, Config.TEST_MASK_DIR, Config.TEST_TIFF_DIR)
    else:
        print("❌ Data missing.")